In [21]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, find_peaks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import SGD, Adam


In [22]:
def preprocess_emg_signal(file_path, fs = 1000, mvc_dict=None):
    df = pd.read_csv(file_path)
    df_processed = df.copy()
    channels = [f'channel{i}' for i in range(1, 9)]
    nyq = 0.5 * fs
    b_band, a_band = butter(4, [20/nyq, 450/nyq], btype='band')
    b_low, a_low = butter(4, 10/nyq, btype='low')

    for ch in channels:
        if ch not in df.columns:
            print(f"Warning: cannot find {ch} in the file!")
            continue
        raw_sig = df[ch].fillna(0).values
        filtered = filtfilt(b_band, a_band, raw_sig)
        rectified = np.abs(filtered)
        envelope = filtfilt(b_low, a_low, rectified)
        envelope = np.maximum(envelope, 0)
        df_processed[ch] = envelope 
    for ch in channels:
        df_processed[ch] = df_processed.groupby('label')[ch].transform(
            lambda x: (x / (x.max() if x.max() > 0 else 1)) * 100
        )

        print(df_processed)

    output_file = file_path.replace('.csv', '_processed.csv')
    df_processed.to_csv(output_file, index=False)

    print("Finishing Preprocessing dataset")

    return df_processed


In [23]:
file_path = r"datasets\train\EMG-data.csv"
df = preprocess_emg_signal(file_path)
df 

          time  channel1  channel2      channel3  channel4  channel5  \
0            1  0.505663  0.000003  0.000000e+00  0.000004  0.000000   
1            5  0.540232  0.000003  4.408627e-07  0.000005  0.000000   
2            6  0.574307  0.000004  1.715085e-06  0.000006  0.000000   
3            7  0.607811  0.000005  2.980392e-06  0.000007  0.000000   
4            8  0.640667  0.000005  4.234796e-06  0.000008  0.000000   
...        ...       ...       ...           ...       ...       ...   
4237902  50962  1.788038  0.000005  7.649281e-06  0.000008  0.000035   
4237903  50963  1.784848  0.000005  7.643663e-06  0.000008  0.000035   
4237904  50964  1.782188  0.000005  7.638959e-06  0.000008  0.000035   
4237905  50965  1.780000  0.000005  7.635074e-06  0.000008  0.000035   
4237906  50966  1.778226  0.000005  7.631911e-06  0.000008  0.000035   

             channel6      channel7      channel8  class  label  
0        4.586132e-07  1.549889e-07  0.000000e+00      0      1  
1  

,time,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,class,label
0,1,0.505663,0.426684,0.000000,0.569724,0.000000,0.098621,0.035534,0.000000,0,1
1,5,0.540232,0.521757,0.077848,0.747488,0.000000,0.207517,0.066253,0.000000,0,1
2,6,0.574307,0.618733,0.302852,0.923943,0.000000,0.314503,0.097629,0.000000,0,1
3,7,0.607811,0.717485,0.526282,1.098786,0.000000,0.419383,0.129625,0.000000,0,1
4,8,0.640667,0.817878,0.747786,1.271718,0.000000,0.521969,0.162198,0.008943,0,1
...,...,...,...,...,...,...,...,...,...,...,...
4237902,50962,1.788038,1.957604,4.085815,1.574949,5.676281,4.573566,3.862965,4.831329,0,36
4237903,50963,1.784848,1.957495,4.082814,1.576233,5.682195,4.571260,3.867166,4.830561,0,36
4237904,50964,1.782188,1.957413,4.080302,1.577300,5.687095,4.569356,3.870658,4.829887,0,36
4237905,50965,1.780000,1.957352,4.078227,1.578175,5.691099,4.567802,3.873519,4.829305,0,36


In [24]:
def create_sliding_windows(data, window_size, step_size, channels):
    X, y = [], []
    for (subj, cls_val), group in data.groupby(['label', 'class']):
        group_vals = group[channels].values

        for i in range(0, len(group_vals) - window_size + 1, step_size):
            window = group_vals[i : i + window_size]
            X.append(window)
            y.append(cls_val)
            
    return np.array(X), np.array(y)

In [25]:
def prepare_emg_data_for_cnn(csv_file, window_size=200, step_size=50):
    df = pd.read_csv(csv_file)
    df = df[df['class'] != 0]
    df['class'] = df['class'] - 1

    train_subjects = list(range(1, 31))
    test_subjects = list(range(31, 37))

    train_df = df[df['label'].isin(train_subjects)]
    test_df = df[df['label'].isin(test_subjects)]

    channels = [f'channel{i}' for i in range(1, 9)]

    print("Slicing Training set")
    X_train, y_train = create_sliding_windows(train_df, window_size, step_size, channels)

    print("Slicing Testing set")
    X_test, y_test = create_sliding_windows(test_df, window_size, step_size, channels)

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape:  {X_test.shape}")
    print(f"y_test shape:  {y_test.shape}")

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = prepare_emg_data_for_cnn('datasets/train/EMG-data_processed.csv', window_size=200, step_size=50)

Slicing Training set
Slicing Testing set
X_train shape: (24726, 200, 8)
y_train shape: (24726,)
X_test shape:  (4769, 200, 8)
y_test shape:  (4769,)


In [27]:
def build_and_train_cnn(X_train, y_train, X_test, y_test, num_classes=7):
    input_shape = (X_train.shape[1], X_train.shape[2])
    model = Sequential([
        Conv1D(filters=32, kernel_size=3, kernel_regularizer=tf.keras.regularizers.l2(0.01), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        Flatten(),
        Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dropout(0.7),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

    print("============== Model Summary: ==============")
    model.summary()

    print("Starting Training......")

    history = model.fit(X_train, y_train, 
                        epochs=30,       
                        batch_size=64,      
                        validation_data=(X_test, y_test))
    test_loss, test_acc = model.evaluate(X_test, y_test)
    print(f"Test accuracy: {test_acc * 100:.2f}%")
    
    return model, history

model, history = build_and_train_cnn(X_train, y_train, X_test, y_test, num_classes=7)

============== Model Summary: ==============


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_8 (Conv1D)               │ (None, 198, 32)        │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 198, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 99, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 97, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 97, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       393,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 401,639 (1.53 MB)

 Trainable params: 401,447 (1.53 MB)

 Non-trainable params: 192 (768.00 B)

Starting Training......
Epoch 1/30
387/387 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6915 - loss: 3.3390 - val_accuracy: 0.8830 - val_loss: 1.1532
Epoch 2/30
387/387 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.7934 - loss: 1.2043 - val_accuracy: 0.8819 - val_loss: 0.6557
Epoch 3/30
387/387 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8201 - loss: 0.8031 - val_accuracy: 0.8696 - val_loss: 0.6095
Epoch 4/30
387/387 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8271 - loss: 0.7250 - val_accuracy: 0.8784 - val_loss: 0.5763
Epoch 5/30
387/387 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.8363 - loss: 0.7051 - val_accuracy: 0.8815 - val_loss: 0.5718
Epoch 6/30
387/387 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.8330 - loss: 0.6905 - val_accuracy: 0.8792 - val_loss: 0.5646
Epoch 7/30
387/387 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.8438 - loss: 0.6577 - val_accuracy: 0.8863 - val_loss: 0.5523
Epoch 8/30
387/387 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.8483 -